## Unit 2: Web Application Development with Flask & Databases
### 📘 Module 02: Flask Framework Architecture, Dynamic Routing, Templates & Blueprints

---

> **Target Audience:** Master of Science in Information Technology (MSc IT)  
> **Prerequisites:** Python Fundamentals (OOP, Decorators, Dictionaries), Basic HTML/HTTP Concepts

---

### 🎯 Bloom's Taxonomy Learning Objectives

By the end of this laboratory lecture, students will be able to:
1. **Remember & Understand:** State the role of the Web Server Gateway Interface (**WSGI** - PEP 3333) and explain how Flask's micro-framework core differs from monolithic architectures (Django) and asynchronous frameworks (FastAPI).
2. **Apply:** Construct robust web routes using dynamic URL converters (`<int:id>`, `<string:code>`, `<path:subpath>`) and generate deterministic URLs via `url_for()`.
3. **Analyze:** Deconstruct HTTP requests using the Flask `request` context (`request.args` for query parameters, `request.form` for form data, `request.get_json()` for APIs).
4. **Evaluate:** Design modular, maintainable web applications by separating concerns through **Jinja2 Template Inheritance** and **Flask Blueprints**.
5. **Create:** Implement and test an interactive academic web service with input validation and clean error handling.

---

### 💡 Interactive Notebook Guide: How We Run Flask in Jupyter
In standard terminal workflows, running `app.run()` launches an active HTTP socket server that stays running until terminated (`Ctrl+C`). In Jupyter notebooks, invoking `app.run()` would block the notebook kernel and prevent you from running subsequent cells!




In [ ]:
# Cell 1: Environment Verification
# Let's verify that Python 3.11+, Flask, and Jinja2 are installed and ready.
import sys
import importlib.metadata
import flask
import jinja2

werkzeug_ver = importlib.metadata.version("werkzeug")

print(f"🐍 Python Version   : {sys.version.split()[0]}")
print(f"🌶️ Flask Version    : {flask.__version__}")
print(f"🎨 Jinja2 Version   : {jinja2.__version__}")
print(f"⚙️ Werkzeug Version : {werkzeug_ver}")
print("✅ Environment is fully configured for MSc IT Web Development!")


🐍 Python Version   : 3.13.7
🌶️ Flask Version    : 3.1.3
🎨 Jinja2 Version   : 3.1.6
⚙️ Werkzeug Version : 3.1.8
✅ Environment is fully configured for MSc IT Web Development!


<ipython-input-1-f6df2bd5864a>:11: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"🌶️ Flask Version    : {flask.__version__}")


---
## 🌐 1. Modern Web Architecture, HTTP & The WSGI Standard

Before writing web code, an MSc IT student must understand what happens under the hood when a client connects to a Python web application.

### 1.1 The Client-Server HTTP Paradigm
When a user accesses `http://127.0.0.1:5000/students?dept=CS`:
1. The **Browser (Client)** opens a TCP connection and transmits an HTTP Request message:
   ```http
   GET /students?dept=CS HTTP/1.1
   Host: 127.0.0.1:5000
   User-Agent: Mozilla/5.0
   Accept: text/html,application/xhtml+xml
   ```
2. The **Server** processes the request, locates the resource or executes code, and responds with an HTTP Response message:
   ```http
   HTTP/1.1 200 OK
   Content-Type: text/html; charset=utf-8
   Content-Length: 482

   <!DOCTYPE html><html>...</html>
   ```

### 1.2 What is WSGI (PEP 3333)?
A raw web server (like Apache, Nginx, or Werkzeug's dev server) knows how to handle TCP sockets, SSL/TLS, and raw HTTP text. However, Python functions do not inherently know how to speak raw network sockets.

**WSGI (Web Server Gateway Interface)** is a standardized interface between Python web applications and web servers.
A WSGI application is simply a Python callable (such as an instance of `Flask`) that accepts two parameters:
```python
def application(environ, start_response):
    # environ: dictionary containing request headers, query strings, paths
    # start_response: callable used to set status code and HTTP headers
    start_response('200 OK', [('Content-Type', 'text/plain')])
    return [b"Hello from raw WSGI!"]
```

Flask implements this WSGI callable signature inside `Flask.__call__`.


### 1.3 Architectural Comparison: Flask vs. Django vs. FastAPI

| Feature / Metric | 🌶️ Flask | 🎸 Django | ⚡ FastAPI |
| :--- | :--- | :--- | :--- |
| **Paradigm** | Micro-framework (Werkzeug + Jinja2) | Full-Stack ("Batteries Included") | Modern Asynchronous API Engine |
| **Philosophy** | Explicit is better than implicit; minimal core | Convention over configuration; everything built-in | High performance, type-hint driven, auto-docs |
| **Database ORM** | Bring Your Own (SQLAlchemy, Peewee, raw SQL) | Built-in Django ORM | Bring Your Own (SQLAlchemy 2.0, SQLModel) |
| **Admin Panel** | Optional extension (Flask-Admin) | Built-in production-ready Admin | None (custom or external third-party) |
| **Form Handling** | WTForms (via Flask-WTF) | Built-in Form & ModelForm validation | Pydantic data models & request bodies |
| **Best Used For** | Microservices, learning internals, custom architectures | Large enterprise portals, content CMS, monolithic backends | High-concurrency REST/GraphQL APIs, ML model serving |


---
## 🔬 2. Anatomy of a Flask Application & Request-Response Lifecycle

Let us dissect the foundational components of every Flask application.

### 2.1 The Core Components
```python
from flask import Flask

app = Flask(__name__)
```

- **`Flask` class:** The central application object. It manages configuration, route tables, blueprints, template folders, and hooks.
- **`__name__` argument:** A special Python variable denoting the current module name. When executed directly, `__name__ == "__main__"`. Flask uses this module path to calculate the root directory of your project, locate static assets (`static/`), and find templates (`templates/`).

### 2.2 The Decorator Pattern: `@app.route()`
In Python, a decorator wraps a function and modifies its behavior. 
When you write:
```python
@app.route("/")
def home():
    return "Hello World"
```
Flask does **not** execute `home()` when the script runs! Instead, it registers an internal rule mapping the URL path `'/'` to the callback function `home` inside the routing table (`app.url_map`).


### 2.3 Live Interactive Code: Building and Testing Our First Application
Let's build a clean application instance and test it using Flask's `test_client()`. Notice how we can inspect the HTTP Status Code, Content-Type headers, and response text immediately!


In [ ]:
# Cell 2: First Flask App & In-Notebook HTTP Testing
from flask import Flask

# 1. Instantiate the application
app = Flask(__name__)

# 2. Define the root route
@app.route("/")
def home():
    return "<h1>🎓 Welcome to MSc IT Flask Lab</h1><p>Status: Server Operational</p>"

# 3. Define an informational route
@app.route("/about")
def about():
    return {
        "course": "MSc (IT) Hands-On Python",
        "unit": 2,
        "module": "Flask Framework Fundamentals",
        "framework": "Flask 3.x",
        "status": "Active"
    }

# 4. Test the routes using Flask's native test_client()
with app.test_client() as client:
    # Test Root Route
    res_home = client.get("/")
    print("=== Testing Route: '/' ===")
    print(f"Status Code  : {res_home.status_code}")
    print(f"Content-Type : {res_home.content_type}")
    print(f"Body Content :\n{res_home.data.decode('utf-8')}\n")

    # Test About Route (Returns JSON)
    res_about = client.get("/about")
    print("=== Testing Route: '/about' ===")
    print(f"Status Code  : {res_about.status_code}")
    print(f"Content-Type : {res_about.content_type}")
    print(f"JSON Output  : {res_about.get_json()}")


=== Testing Route: '/' ===
Status Code  : 200
Content-Type : text/html; charset=utf-8
Body Content :
<h1>🎓 Welcome to MSc IT Flask Lab</h1><p>Status: Server Operational</p>

=== Testing Route: '/about' ===
Status Code  : 200
Content-Type : application/json
JSON Output  : {'course': 'MSc (IT) Hands-On Python', 'framework': 'Flask 3.x', 'module': 'Flask Framework Fundamentals', 'status': 'Active', 'unit': 2}


---
## 🗺️ 3. URL Dispatching, Routing Table & Dynamic Converters

Modern web applications do not use static URLs like `/profile.php?id=42`. They use clean, expressive, RESTful resource paths like `/students/101` or `/courses/CS501`.

### 3.1 How Routing Works: `app.url_map`
Every Flask instance maintains a `Map` containing `Rule` objects. When a request arrives, Flask checks the requested URL path against these rules in order.

### 3.2 Dynamic URL Variable Converters
Flask provides built-in type converters that automatically validate URL fragments and parse them into native Python data types before passing them as arguments to your view function:

| Converter Syntax | Data Type Extracted | Valid Input Example | Rejected Input Example (404) |
| :--- | :--- | :--- | :--- |
| `<string:variable>` *(default)* | `str` (no slashes) | `/user/john` | `/user/john/smith` |
| `<int:variable>` | `int` (positive integer) | `/students/101` | `/students/abc`, `/students/-5` |
| `<float:variable>` | `float` (floating-point) | `/temperature/98.6` | `/temperature/high` |
| `<path:variable>` | `str` (accepts `/` slashes) | `/files/notes/unit2.pdf` | (Accepts all valid path strings) |
| `<uuid:variable>` | `uuid.UUID` object | `/tx/550e8400-e29b-41d4-a716-446655440000` | `/tx/12345` |



### 3.3 Reverse URL Generation with `url_for()`
> [!IMPORTANT]
> **Professor's Rule of Clean Architecture:** **Never hardcode URLs** in your view functions or HTML templates!  
> If you hardcode `<a href="/students/101">`, and tomorrow the department renames the URL prefix to `/portal/students/101`, every single hardcoded link in your project will break!
>
> Instead, use `url_for(endpoint, **values)`:
> ```python
> url_for('get_student', student_id=101)  # Produces '/students/101'
> ```
> `url_for()` resolves the URL dynamically based on the function name, automatically handles URL escaping, and attaches any extra keyword arguments as query parameters (e.g. `url_for('get_student', student_id=101, tab='grades')` -> `'/students/101?tab=grades'`).

### 3.5 Live Code: Realistic Student Registry & Dynamic Converters


In [ ]:
# Cell 3: Dynamic Routing with Realistic Academic Dataset
from flask import Flask, url_for, abort

app_routing = Flask(__name__)

# Realistic Academic Dataset for MSc IT Students
STUDENTS_DB = {
    101: {"name": "Aarav Sharma", "dept": "Computer Science", "gpa": 3.85, "semester": 3},
    102: {"name": "Diya Patel", "dept": "Information Technology", "gpa": 3.92, "semester": 3},
    103: {"name": "Rohan Verma", "dept": "Data Science", "gpa": 3.40, "semester": 1},
}

COURSES_DB = {
    "CS501": {"title": "Advanced Python Architecture", "credits": 4},
    "IT502": {"title": "Distributed Cloud Systems", "credits": 3},
    "DS503": {"title": "Machine Learning Engineering", "credits": 4}
}

# 1. Integer converter route: /students/<int:student_id>
@app_routing.route("/students/<int:student_id>")
def get_student(student_id):
    student = STUDENTS_DB.get(student_id)
    if not student:
        return {"error": f"Student with ID {student_id} not found in university registry"}, 404
    return {
        "status": "success",
        "student_id": student_id,
        "record": student
    }

# 2. String converters with multiple path parameters
@app_routing.route("/departments/<string:dept_code>/courses/<string:course_code>")
def get_department_course(dept_code, course_code):
    course = COURSES_DB.get(course_code.upper())
    if not course:
        return {"error": f"Course code {course_code} not offered under department {dept_code}"}, 404
    return {
        "department": dept_code.upper(),
        "course_code": course_code.upper(),
        "details": course
    }

# 3. Path converter: allows slashes in the variable
@app_routing.route("/documents/<path:doc_path>")
def get_document(doc_path):
    return {"message": f"Serving institutional document from path: /{doc_path}"}

# Testing the dynamic routing engine
with app_routing.test_client() as client:
    print("--- 1. Valid Student Query (ID: 101) ---")
    r1 = client.get("/students/101")
    print(f"Status: {r1.status_code} | Data: {r1.get_json()}")

    print("\n--- 2. Non-existent Student Query (ID: 999) ---")
    r2 = client.get("/students/999")
    print(f"Status: {r2.status_code} | Data: {r2.get_json()}")

    print("\n--- 3. Invalid Type Parameter ('/students/abc') ---")
    # Flask converter fails at regex matching and automatically emits 404!
    r3 = client.get("/students/abc")
    print(f"Status: {r3.status_code} (Rejected by <int:> converter automatically!)")

    print("\n--- 4. Multi-variable Route ---")
    r4 = client.get("/departments/CS/courses/CS501")
    print(f"Status: {r4.status_code} | Data: {r4.get_json()}")

    print("\n--- 5. Path Converter with Nested Subdirectories ---")
    r5 = client.get("/documents/syllabus/2026/msc_it_unit2.pdf")
    print(f"Status: {r5.status_code} | Data: {r5.get_json()}")

    print("\n--- 6. Reverse URL Generation with url_for() ---")
    with app_routing.test_request_context():
        print("Generated URL for student 102 :", url_for("get_student", student_id=102))
        print("Generated URL with query args :", url_for("get_student", student_id=102, view="transcript", term="fall"))


--- 1. Valid Student Query (ID: 101) ---
Status: 200 | Data: {'record': {'dept': 'Computer Science', 'gpa': 3.85, 'name': 'Aarav Sharma', 'semester': 3}, 'status': 'success', 'student_id': 101}

--- 2. Non-existent Student Query (ID: 999) ---
Status: 404 | Data: {'error': 'Student with ID 999 not found in university registry'}

--- 3. Invalid Type Parameter ('/students/abc') ---
Status: 404 (Rejected by <int:> converter automatically!)

--- 4. Multi-variable Route ---
Status: 200 | Data: {'course_code': 'CS501', 'department': 'CS', 'details': {'credits': 4, 'title': 'Advanced Python Architecture'}}

--- 5. Path Converter with Nested Subdirectories ---
Status: 200 | Data: {'message': 'Serving institutional document from path: /syllabus/2026/msc_it_unit2.pdf'}

--- 6. Reverse URL Generation with url_for() ---
Generated URL for student 102 : /students/102
Generated URL with query args : /students/102?view=transcript&term=fall


---
## 📨 4. HTTP Methods & Dissecting the Flask `request` Object

Web applications interact with clients using standard HTTP methods. Understanding the semantic difference between them is vital for designing robust web forms and REST APIs.

### 4.1 Common HTTP Methods Explained

| Method | Idempotent? | Safe? | Typical Purpose | Where Data is Transmitted |
| :--- | :---: | :---: | :--- | :--- |
| **`GET`** | ✅ Yes | ✅ Yes | Retrieve existing resource or view a page | In the URL query string (`?key=value`) |
| **`POST`** | ❌ No | ❌ No | Create a new resource or submit an HTML form | In the HTTP Request Body |
| **`PUT`** | ✅ Yes | ❌ No | Replace an entire existing resource | In the HTTP Request Body |
| **`DELETE`** | ✅ Yes | ❌ No | Delete an existing resource | In the URL path or Request Body |

> [!NOTE]
> **Safe:** Safe methods do not modify server state (reading data).  
> **Idempotent:** Making the same request multiple times produces the exact same server state as making it once.

### 4.2 The Flask `request` Context Proxy
Flask exposes a thread-safe context proxy named `request`. Depending on how data is sent by the client, you access it via different properties:

### 4.3 Key `request` Attributes Cheat-Sheet
- `request.method`: The HTTP verb as a string (e.g. `'GET'`, `'POST'`).
- `request.args.get('key', default_value)`: Extracts URL query parameters. Always use `.get()` to avoid `KeyError` exceptions when an optional parameter is missing!
- `request.form.get('key')`: Extracts submitted form field values.
- `request.get_json(silent=True)`: Parses JSON payload into a Python dictionary.
- `request.headers.get('User-Agent')`: Reads HTTP headers sent by the client.

### 4.4 Live Code: Student Enrollment System (Handling GET & POST)


In [ ]:
# Cell 4: Handling Form Submissions and Query Parameters
from flask import Flask, request, jsonify

app_methods = Flask(__name__)

# In-memory database of registered students
COURSE_ENROLLMENTS = [
    {"student_id": 101, "course_code": "CS501", "grade": "A"},
    {"student_id": 102, "course_code": "CS501", "grade": "A-"},
]

@app_methods.route("/enrollments", methods=["GET", "POST"])
def manage_enrollments():
    # -------------------------------------------------------------
    # CASE 1: GET Request -> Query/Filter Existing Records
    # -------------------------------------------------------------
    if request.method == "GET":
        # Extract query parameters from URL: e.g., /enrollments?course=CS501
        course_filter = request.args.get("course")
        
        if course_filter:
            filtered = [e for e in COURSE_ENROLLMENTS if e["course_code"] == course_filter.upper()]
            return {
                "filter_applied": {"course": course_filter.upper()},
                "count": len(filtered),
                "enrollments": filtered
            }, 200
        
        return {
            "total_count": len(COURSE_ENROLLMENTS),
            "enrollments": COURSE_ENROLLMENTS
        }, 200

    # -------------------------------------------------------------
    # CASE 2: POST Request -> Process New Enrollment Submission
    # -------------------------------------------------------------
    elif request.method == "POST":
        # Support both form submissions and JSON API payloads
        data = request.form if request.form else request.get_json(silent=True)
        
        if not data:
            return {"error": "Missing submission payload. Provide form-data or JSON."}, 400
        
        student_id = data.get("student_id")
        course_code = data.get("course_code")
        
        # Server-Side Validation
        if not student_id or not course_code:
            return {"error": "Both 'student_id' and 'course_code' are required fields!"}, 422
        
        try:
            student_id = int(student_id)
        except ValueError:
            return {"error": "Field 'student_id' must be an integer."}, 422
            
        # Check for duplicate enrollment
        duplicate = any(e["student_id"] == student_id and e["course_code"] == course_code.upper() for e in COURSE_ENROLLMENTS)
        if duplicate:
            return {"error": f"Student {student_id} is already enrolled in {course_code.upper()}!"}, 409

        new_record = {
            "student_id": student_id,
            "course_code": course_code.upper(),
            "grade": "Pending"
        }
        COURSE_ENROLLMENTS.append(new_record)
        
        return {
            "status": "success",
            "message": f"Student {student_id} successfully enrolled in {course_code.upper()}",
            "record": new_record
        }, 201

# Testing GET and POST requests
with app_methods.test_client() as client:
    print("--- 1. GET: Fetch All Enrollments ---")
    r1 = client.get("/enrollments")
    print("Status:", r1.status_code, "| Data:", r1.get_json())

    print("\n--- 2. POST: Enroll New Student (Form Submission) ---")
    r2 = client.post("/enrollments", data={"student_id": "103", "course_code": "CS501"})
    print("Status:", r2.status_code, "| Data:", r2.get_json())

    print("\n--- 3. POST: Validation Error (Missing Course Code) ---")
    r3 = client.post("/enrollments", data={"student_id": "104"})
    print("Status:", r3.status_code, "| Error:", r3.get_json())

    print("\n--- 4. POST: Duplicate Check (Enrolling Student 101 again) ---")
    r4 = client.post("/enrollments", data={"student_id": "101", "course_code": "CS501"})
    print("Status:", r4.status_code, "| Conflict:", r4.get_json())

    print("\n--- 5. GET with Query Parameter: Filter by Course ---")
    r5 = client.get("/enrollments?course=CS501")
    print("Status:", r5.status_code, "| Filtered:", r5.get_json())


--- 1. GET: Fetch All Enrollments ---
Status: 200 | Data: {'enrollments': [{'course_code': 'CS501', 'grade': 'A', 'student_id': 101}, {'course_code': 'CS501', 'grade': 'A-', 'student_id': 102}], 'total_count': 2}

--- 2. POST: Enroll New Student (Form Submission) ---
Status: 201 | Data: {'message': 'Student 103 successfully enrolled in CS501', 'record': {'course_code': 'CS501', 'grade': 'Pending', 'student_id': 103}, 'status': 'success'}

--- 3. POST: Validation Error (Missing Course Code) ---
Status: 422 | Error: {'error': "Both 'student_id' and 'course_code' are required fields!"}

--- 4. POST: Duplicate Check (Enrolling Student 101 again) ---
Status: 409 | Conflict: {'error': 'Student 101 is already enrolled in CS501!'}

--- 5. GET with Query Parameter: Filter by Course ---
Status: 200 | Filtered: {'count': 3, 'enrollments': [{'course_code': 'CS501', 'grade': 'A', 'student_id': 101}, {'course_code': 'CS501', 'grade': 'A-', 'student_id': 102}, {'course_code': 'CS501', 'grade': 'Pendi

---
## 🎨 5. Jinja2 Templating Engine & Dynamic HTML Rendering

Returning raw HTML strings concatenated with Python strings (e.g. `f"<h1>Hello {name}</h1>"`) leads to unmaintainable code and exposes your application to catastrophic **Cross-Site Scripting (XSS)** vulnerabilities!

Flask integrates **Jinja2**, a fast, expressive, and sandboxed template engine.

### 5.1 Jinja2 Delimiters Syntax

| Delimiter | Purpose | Example |
| :--- | :--- | :--- |
| `{{ expression }}` | Evaluates and prints a variable to output | `<h3>Welcome, {{ student.name }}!</h3>` |
| `{% statement %}` | Executes control logic (`if`, `for`, `block`, `extends`) | `{% for c in courses %} <li>{{ c }}</li> {% endfor %}` |
| `{# comment #}` | Internal template comment (never sent to browser) | `{# This is an internal professor note #}` |

### 5.2 Jinja2 Filters
Filters allow transformation of variables inside templates using the pipe (`|`) operator:
- `{{ student.name|title }}`: Capitalizes each word.
- `{{ student.gpa|round(2) }}`: Rounds numbers to 2 decimal places.
- `{{ student.registered_courses|length }}`: Obtains list size.
- `{{ description|default('No syllabus provided') }}`: Provides fallbacks for `None` values.

### 5.3 Template Inheritance Architecture
Template inheritance is one of Jinja2's most powerful features. Instead of copying headers, navigation bars, and footers across 20 HTML files, you define a single `base.html` skeleton containing placeholders (`{% block ... %}`), and individual child templates extend it.

```mermaid
flowchart TD
    BaseLayout["📄 base.html<br>(Master Skeleton: <!DOCTYPE>, &lt;head&gt;, Navbar, Scripts)"]
    BlockContent["📦 {% block content %} ... {% endblock %}"]
    BaseLayout --- BlockContent

    ChildA["📄 student_list.html<br>{% extends 'base.html' %}"]
    ChildB["📄 course_catalog.html<br>{% extends 'base.html' %}"]
    ChildC["📄 grade_report.html<br>{% extends 'base.html' %}"]

    ChildA -->|Fills content block with Table| BlockContent
    ChildB -->|Fills content block with Cards| BlockContent
    ChildC -->|Fills content block with Grade Card| BlockContent

    BlockContent --> FinalRender["🖥️ Compiled HTML Page Rendered to Browser"]
```

### 5.4 Live Code: Dynamic Grade Report & Honor Roll Dashboard
In this demonstration, we use `render_template_string()` to inspect the exact rendered HTML layout and styling without needing external files.


In [ ]:
# Cell 5: Jinja2 Dynamic Rendering with Loops, Conditionals, and Filters
from flask import Flask, render_template_string

app_templates = Flask(__name__)

# Complete HTML Template using Jinja2 syntax
JINJA_DASHBOARD_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{{ university_name }} — MSc IT Grade Report</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background: #f8fafc; color: #1e293b; }
        .card { background: white; border-radius: 8px; padding: 20px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); max-width: 750px; margin: auto; }
        h2 { color: #1e3a8a; border-bottom: 2px solid #e2e8f0; padding-bottom: 10px; }
        table { width: 100%; border-collapse: collapse; margin-top: 15px; }
        th, td { padding: 10px 14px; text-align: left; border-bottom: 1px solid #e2e8f0; }
        th { background-color: #f1f5f9; color: #475569; font-weight: 600; }
        .badge { padding: 4px 10px; border-radius: 9999px; font-size: 0.82rem; font-weight: bold; }
        .badge-honor { background-color: #dcfce7; color: #166534; }
        .badge-good { background-color: #dbeafe; color: #1e40af; }
        .badge-probation { background-color: #fee2e2; color: #991b1b; }
        .summary-box { background: #f8fafc; border-left: 4px solid #3b82f6; padding: 10px 15px; margin-top: 15px; }
    </style>
</head>
<body>
    <div class="card">
        <h2>🏛️ {{ university_name }}</h2>
        <h3>Department of {{ department }} — Semester {{ semester }}</h3>

        <p><strong>Student Name:</strong> {{ student.name|title }} (ID: {{ student.id }})</p>
        <p><strong>Cumulative GPA:</strong> {{ student.gpa|round(2) }}</p>

        <!-- Conditional Badge Logic -->
        <p><strong>Academic Standing:</strong>
            {% if student.gpa >= 3.80 %}
                <span class="badge badge-honor">🌟 Dean's Honor Roll</span>
            {% elif student.gpa >= 3.00 %}
                <span class="badge badge-good">✅ Good Standing</span>
            {% else %}
                <span class="badge badge-probation">⚠️ Academic Probation</span>
            {% endif %}
        </p>

        <!-- Course Iteration Table -->
        <h4>Registered Courses ({{ courses|length }} Total)</h4>
        <table>
            <thead>
                <tr>
                    <th>#</th>
                    <th>Course Code</th>
                    <th>Title</th>
                    <th>Credits</th>
                    <th>Grade</th>
                </tr>
            </thead>
            <tbody>
                {% for course in courses %}
                <tr>
                    <td>{{ loop.index }}</td>
                    <td><strong>{{ course.code }}</strong></td>
                    <td>{{ course.title }}</td>
                    <td>{{ course.credits }}</td>
                    <td>{{ course.grade }}</td>
                </tr>
                {% else %}
                <tr>
                    <td colspan="5" style="text-align: center; color: #94a3b8;">No registered courses found.</td>
                </tr>
                {% endfor %}
            </tbody>
        </table>

        <div class="summary-box">
            <em>Report generated automatically by Flask Jinja2 Templating Engine.</em>
        </div>
    </div>
</body>
</html>
"""

@app_templates.route("/student/report")
def student_report():
    sample_student = {"id": 101, "name": "aarav sharma", "gpa": 3.875}
    sample_courses = [
        {"code": "CS501", "title": "Advanced Python Systems", "credits": 4, "grade": "A"},
        {"code": "IT502", "title": "Cloud Computing Infrastructure", "credits": 3, "grade": "A"},
        {"code": "DS503", "title": "Machine Learning Engineering", "credits": 4, "grade": "A-"},
    ]
    return render_template_string(
        JINJA_DASHBOARD_TEMPLATE,
        university_name="Metropolitan University of Technology",
        department="Information Technology",
        semester=3,
        student=sample_student,
        courses=sample_courses
    )

# Render and test the template
with app_templates.test_client() as client:
    res = client.get("/student/report")
    print(f"Status Code  : {res.status_code}")
    print(f"Content-Type : {res.content_type}")
    print("Rendered HTML Preview (First 25 lines):")
    for line in res.data.decode("utf-8").strip().splitlines()[:25]:
        print("  ", line)


Status Code  : 200
Content-Type : text/html; charset=utf-8
Rendered HTML Preview (First 25 lines):
   <!DOCTYPE html>
   <html lang="en">
   <head>
       <meta charset="UTF-8">
       <title>Metropolitan University of Technology — MSc IT Grade Report</title>
       <style>
           body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background: #f8fafc; color: #1e293b; }
           .card { background: white; border-radius: 8px; padding: 20px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); max-width: 750px; margin: auto; }
           h2 { color: #1e3a8a; border-bottom: 2px solid #e2e8f0; padding-bottom: 10px; }
           table { width: 100%; border-collapse: collapse; margin-top: 15px; }
           th, td { padding: 10px 14px; text-align: left; border-bottom: 1px solid #e2e8f0; }
           th { background-color: #f1f5f9; color: #475569; font-weight: 600; }
           .badge { padding: 4px 10px; border-radius: 9999px; font-size: 0.82rem; font-weight: bold

---
## 🧩 6. Modular Application Architecture with Blueprints

### 6.1 The Monolith Problem
When building small scripts, placing all routes, database handlers, and views inside a single `app.py` is fine. However, in enterprise or large-scale software (e.g. an ERP or University Management System), having hundreds of routes in one file creates massive merge conflicts, tight coupling, and difficult maintenance.

### 6.2 What is a Flask Blueprint?
A **Blueprint** is an organizational pattern for structuring modular Flask applications. Think of a Blueprint as a "sub-application" or reusable package:
- It defines routes, error handlers, and template directories independently.
- It is registered onto the main application instance via `app.register_blueprint()`.
- It allows assigning unique **URL prefixes** (e.g. `/auth`, `/students`, `/admin`) and name isolation (`url_for('auth.login')`).


### 6.3 Live Code: Registering and Isolating Blueprints


In [ ]:
# Cell 6: Creating and Registering Modular Flask Blueprints
from flask import Flask, Blueprint, jsonify

# =============================================================================
# 1. Blueprint: Authentication & Identity (/auth)
# =============================================================================
auth_bp = Blueprint("auth", __name__)

@auth_bp.route("/login", methods=["POST"])
def login():
    return jsonify({"module": "auth", "status": "authenticated", "token": "jwt_token_demo_98765"})

@auth_bp.route("/logout")
def logout():
    return jsonify({"module": "auth", "status": "session_terminated"})


# =============================================================================
# 2. Blueprint: Academic Student Portal (/portal)
# =============================================================================
portal_bp = Blueprint("portal", __name__)

@portal_bp.route("/dashboard")
def dashboard():
    return jsonify({
        "module": "portal",
        "features": ["course_registration", "view_transcript", "fee_receipts"]
    })

@portal_bp.route("/announcements")
def announcements():
    return jsonify({
        "module": "portal",
        "announcements": [
            {"date": "2026-09-10", "title": "MSc IT Midterm Exam Schedule Released"},
            {"date": "2026-09-15", "title": "Flask Web Development Lab Hackathon"}
        ]
    })


# =============================================================================
# 3. Master Application Factory
# =============================================================================
app_modular = Flask(__name__)

# Register Blueprints with explicit URL prefixes
app_modular.register_blueprint(auth_bp, url_prefix="/auth")
app_modular.register_blueprint(portal_bp, url_prefix="/portal")

@app_modular.route("/")
def index():
    return {"message": "University API Gateway Root", "active_blueprints": ["/auth", "/portal"]}

# Test the modular endpoints
with app_modular.test_client() as client:
    print("--- 1. Testing Gateway Root ---")
    print(client.get("/").get_json())

    print("\n--- 2. Testing Auth Blueprint: POST /auth/login ---")
    print(client.post("/auth/login").get_json())

    print("\n--- 3. Testing Portal Blueprint: GET /portal/announcements ---")
    print(client.get("/portal/announcements").get_json())

    print("\n--- 4. Notice URL Name Isolation ---")
    with app_modular.test_request_context():
        print("Reverse URL for auth.login       :", url_for("auth.login"))
        print("Reverse URL for portal.dashboard :", url_for("portal.dashboard"))


--- 1. Testing Gateway Root ---
{'active_blueprints': ['/auth', '/portal'], 'message': 'University API Gateway Root'}

--- 2. Testing Auth Blueprint: POST /auth/login ---
{'module': 'auth', 'status': 'authenticated', 'token': 'jwt_token_demo_98765'}

--- 3. Testing Portal Blueprint: GET /portal/announcements ---
{'announcements': [{'date': '2026-09-10', 'title': 'MSc IT Midterm Exam Schedule Released'}, {'date': '2026-09-15', 'title': 'Flask Web Development Lab Hackathon'}], 'module': 'portal'}

--- 4. Notice URL Name Isolation ---
Reverse URL for auth.login       : /auth/login
Reverse URL for portal.dashboard : /portal/dashboard


---
## 🚀 7. Deployment Architecture: Development vs. Production

When moving from a local Jupyter lab environment to a live web server, you must run Flask as a standalone process.

### 7.1 The Standalone Entrypoint Pattern
```python
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
```
- **`host='0.0.0.0'`**: Listens on all available network interfaces, allowing other machines on the local network to access your server.
- **`port=5000`**: Default TCP port for Flask.
- **`debug=True`**: Enables the **Werkzeug interactive debugger** and the auto-reloader (restarts the process whenever you edit code).

> [!CAUTION]
> ### 🛑 Critical Security Warning: NEVER use `debug=True` in Production!
> In debug mode, if an unhandled Python exception occurs, Werkzeug renders an interactive Python console directly in the browser! Anyone visiting your site could type `import os; os.system('rm -rf /')` and achieve **Arbitrary Remote Code Execution (RCE)** on your server.
> Always set `debug=False` or disable it via environment variables before deploying.

### 7.2 Development vs Production Architecture Diagram

```mermaid
flowchart TD
    subgraph DevStack ["🛠️ Development Mode (Local Machine)"]
        BrowserDev["🌐 Browser"] --> WerkzeugDev["⚡ Werkzeug Built-in Server<br>(Single-threaded / Not hardened / debug=True)"]
        WerkzeugDev --> FlaskDev["🌶️ Flask App"]
    end

    subgraph ProdStack ["🏢 Production Mode (Enterprise Cloud)"]
        Internet["🌍 Public Internet Traffic"] --> Nginx["🛡️ Nginx / Cloudflare<br>(Reverse Proxy / SSL / Static File Cache / Rate Limiting)"]
        Nginx -->|UNIX Socket or HTTP:8000| Gunicorn["⚙️ Gunicorn / uWSGI<br>(Pre-fork WSGI Master Process)"]
        Gunicorn --> Worker1["👷 Worker 1 (Flask Instance)"]
        Gunicorn --> Worker2["👷 Worker 2 (Flask Instance)"]
        Gunicorn --> Worker3["👷 Worker 3 (Flask Instance)"]
        Gunicorn --> Worker4["👷 Worker 4 (Flask Instance)"]
    end
```

### 7.3 How to Run from the Terminal
In your terminal, navigate to your project directory and run:
```bash
# Option 1: Direct Python invocation
python app.py

# Option 2: Using the modern Flask CLI tool (Recommended)
export FLASK_APP=app.py
export FLASK_DEBUG=1
flask run --host=0.0.0.0 --port=5000
```


---
## 👨‍🏫 8. Professor's Lecture Companion & Discussion Prompts

This section is prepared specifically for faculty and instructors conducting MSc IT classroom lectures and laboratory sessions.

### 8.1 Socratic Discussion Questions for the Classroom

1. **Question:** *What happens if two `@app.route()` decorators define the exact same path (e.g. `'/'`)?*  
   **Professor's Answer:** Flask registers both in `app.url_map`, but evaluates rules top-down. The first registered view function will always be executed; the second will be shadowed unless they specify different HTTP `methods`.
2. **Question:** *Why is `from flask import request` imported as a global variable, yet safe in multi-threaded environments?*  
   **Professor's Answer:** `request` is a **LocalProxy** object. Under the hood, it uses Python's `contextvars` (or thread-local storage in older versions) to resolve dynamically to the request object of the specific thread/coroutine currently executing.
3. **Question:** *What is the fundamental difference between `return render_template(...)` vs `return jsonify(...)`?*  
   **Professor's Answer:** `render_template` compiles HTML with `Content-Type: text/html; charset=utf-8` intended for human web browsers. `jsonify` serializes Python data structures to JSON with `Content-Type: application/json` intended for automated clients, mobile apps, or frontend frameworks (React/Vue).

### 8.2 Common Pitfalls & Debugging Checklist

| HTTP Status Code | Common Cause in Flask | How to Resolve |
| :--- | :--- | :--- |
| **`404 Not Found`** | URL typo, missing trailing slash, or dynamic converter type mismatch (e.g. sending string to `<int:>`). | Check `app.url_map`, verify route parameters match argument names. |
| **`405 Method Not Allowed`** | Sending a `POST` request to a route that only declared `GET` by default. | Add `methods=["GET", "POST"]` to the `@app.route()` decorator. |
| **`400 Bad Request`** | Attempting to access `request.form['key']` directly when the form field was omitted by the user. | Use `request.form.get('key')` which returns `None` rather than crashing with a `KeyError`. |
| **`500 Internal Server Error`** | Unhandled Python exception inside the view function (e.g. `ZeroDivisionError`, `AttributeError`). | Check terminal traceback or enable debug mode in local development. |


---
## 🏋️ 9. Hands-On Exercises & Self-Grading Test Suite

Test your mastery of Flask web development! Complete the three exercises below and run the grading cell to verify your implementation.

### Exercise Requirements:
1. **Exercise 1 (Dynamic Route):** Implement a route `/calculate_gpa/<float:marks>` that accepts student percentage marks (0.0 to 100.0) and returns a JSON object `{"marks": marks, "gpa": marks / 25.0, "result": "Pass" if marks >= 40 else "Fail"}`.
2. **Exercise 2 (Course Registration):** Implement a route `/register_course` supporting both `GET` and `POST`:
   - `GET`: Returns `{"action": "Please submit course_id and student_id via POST"}`.
   - `POST`: Reads `course_id` and `student_id` from `request.form`. Validates that both are present. If either is missing, returns `{"error": "Missing parameters"}`, 400. If valid, returns `{"status": "Registered", "course_id": course_id, "student_id": student_id}`, 201.
3. **Exercise 3 (Jinja2 Honor List):** Implement `/deans_list` which renders an HTML string listing student names who have a GPA >= 3.75 using Jinja2 loop syntax.


In [ ]:
# Cell 7: Student Exercise Workspace
from flask import Flask, request, jsonify, render_template_string

exercise_app = Flask(__name__)

# =============================================================================
# EXERCISE 1: Dynamic Route for GPA Calculation
# =============================================================================
@exercise_app.route("/calculate_gpa/<float:marks>")
def calculate_gpa(marks):
    # TODO: Implement calculation
    gpa = round(marks / 25.0, 2)
    result = "Pass" if marks >= 40.0 else "Fail"
    return jsonify({
        "marks": marks,
        "gpa": gpa,
        "result": result
    })


# =============================================================================
# EXERCISE 2: Course Registration (GET and POST)
# =============================================================================
@exercise_app.route("/register_course", methods=["GET", "POST"])
def register_course():
    if request.method == "GET":
        return jsonify({"action": "Please submit course_id and student_id via POST"}), 200
    
    # POST handling
    course_id = request.form.get("course_id")
    student_id = request.form.get("student_id")
    
    if not course_id or not student_id:
        return jsonify({"error": "Missing parameters"}), 400
        
    return jsonify({
        "status": "Registered",
        "course_id": course_id,
        "student_id": student_id
    }), 201


# =============================================================================
# EXERCISE 3: Jinja2 Dean's Honor Roll Renderer
# =============================================================================
HONOR_ROLL_TEMPLATE = """
<h2>🌟 Dean's Honor Roll</h2>
<ul>
{% for s in students %}
    {% if s.gpa >= 3.75 %}
        <li><strong>{{ s.name }}</strong> — GPA: {{ s.gpa }}</li>
    {% endif %}
{% endfor %}
</ul>
"""

@exercise_app.route("/deans_list")
def deans_list():
    students_data = [
        {"name": "Aarav Sharma", "gpa": 3.85},
        {"name": "Diya Patel", "gpa": 3.92},
        {"name": "Rohan Verma", "gpa": 3.40}
    ]
    return render_template_string(HONOR_ROLL_TEMPLATE, students=students_data)

print("✅ Student exercise functions defined and ready for testing!")


✅ Student exercise functions defined and ready for testing!


In [ ]:
# Cell 8: Automated Self-Grading Verification Test Suite
with exercise_app.test_client() as client:
    passed = 0
    total = 5

    print("🧪 Running Automated Self-Grading Suite for MSc IT Lab...\n")

    # Test 1: GPA Calculator
    res1 = client.get("/calculate_gpa/85.5")
    assert res1.status_code == 200, f"Test 1 Failed: Status {res1.status_code}"
    data1 = res1.get_json()
    assert data1["gpa"] == 3.42 and data1["result"] == "Pass", f"Test 1 Data Mismatch: {data1}"
    print("✅ Test 1 Passed: Dynamic Float Route /calculate_gpa/85.5 correctly returned GPA 3.42 Pass")
    passed += 1

    # Test 2: Course Registration GET
    res2 = client.get("/register_course")
    assert res2.status_code == 200, f"Test 2 Failed: Status {res2.status_code}"
    print("✅ Test 2 Passed: GET /register_course returns instructional prompt")
    passed += 1

    # Test 3: Course Registration POST Success
    res3 = client.post("/register_course", data={"course_id": "CS501", "student_id": "101"})
    assert res3.status_code == 201, f"Test 3 Failed: Status {res3.status_code}"
    data3 = res3.get_json()
    assert data3["status"] == "Registered" and data3["course_id"] == "CS501", f"Test 3 Data Mismatch: {data3}"
    print("✅ Test 3 Passed: POST /register_course with valid parameters returns HTTP 201 Registered")
    passed += 1

    # Test 4: Course Registration POST Validation Error
    res4 = client.post("/register_course", data={"course_id": "CS501"})
    assert res4.status_code == 400, f"Test 4 Failed: Status {res4.status_code}"
    print("✅ Test 4 Passed: POST /register_course with missing field correctly rejected with HTTP 400")
    passed += 1

    # Test 5: Jinja2 Dean's List Rendering
    res5 = client.get("/deans_list")
    assert res5.status_code == 200, f"Test 5 Failed: Status {res5.status_code}"
    html5 = res5.data.decode("utf-8")
    assert "Aarav Sharma" in html5 and "Diya Patel" in html5 and "Rohan Verma" not in html5, "Test 5 HTML Filter Mismatch!"
    print("✅ Test 5 Passed: Jinja2 conditional filter correctly isolated GPA >= 3.75 students")
    passed += 1

    print(f"\n🎉 Outstanding! Passed {passed}/{total} tests! (100% Score)")


🧪 Running Automated Self-Grading Suite for MSc IT Lab...

✅ Test 1 Passed: Dynamic Float Route /calculate_gpa/85.5 correctly returned GPA 3.42 Pass
✅ Test 2 Passed: GET /register_course returns instructional prompt
✅ Test 3 Passed: POST /register_course with valid parameters returns HTTP 201 Registered
✅ Test 4 Passed: POST /register_course with missing field correctly rejected with HTTP 400
✅ Test 5 Passed: Jinja2 conditional filter correctly isolated GPA >= 3.75 students

🎉 Outstanding! Passed 5/5 tests! (100% Score)
